# LIMPIEZA DE DATOS — TIPOS DE CAMBIO
### Tipos de cambio: Peseta (1959-1998) y Euro (1999-2026)
### Incluye:
- Librerías de python utilizadas
- Exploración de fuentes y verificación de estructura
- Carga de datasets
- Conversión de tipo de datos
- Limpieza de datos
- Unión de series

---
**Fuentes:**
- Banco de España (BdE): *Tipos de cambio de la peseta frente a las monedas más relevantes (1959-1998)*. Versión 1. Datos mensuales (cuadro_01) y diarios (cuadro_02). Unidad: pesetas por unidad de moneda extranjera.
- Banco Central Europeo (BCE): *Euro foreign exchange reference rates (1999-2026)*. Frecuencia: diaria (días hábiles). Unidad: unidades de moneda extranjera por 1 EUR.

**Nota sobre convenciones de cotización:**
- BdE (peseta): cotización **directa** → X pesetas por 1 unidad extranjera (ej. 142 USD/pts, 142 pesetas por dólar)
- BCE (euro): cotización **indirecta** → X unidades extranjeras por 1 EUR (ej. 1.17 EUR/USD, 1.17 dólares por euro)
- Se convierte la serie en pesetas del BdE (cotización directa, pts por divisa) a cotización indirecta (divisa por EUR), convención estándar del BCE, para empalmar con las series modernas y unificar el dataset
- La conversión entre pesetas y euros se realiza mediante el tipo de conversión fijo peseta/euro (166,386 pts/EUR).

# 0. Importación de "Librerías"

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# 1. Exploración y carga de datasets

In [2]:
ruta_bde  = '../Fuentes/BDE/Tipos_de_cambio/'
ruta_bce  = '../Fuentes/BDE/Tipos_de_cambio/'

### 1.1 Exploración del archivo BdE — Tipos de cambio de la peseta (xlsx)
El archivo contiene **5 hojas**: Índice de hojas, Índice de series, cuadro_01 (mensual), cuadro_02 (diario) y Notas.
Trabajaremos con **cuadro_01** (mensual, 1959-1998) que es la frecuencia compatible con el resto de la base de datos.

In [3]:
df_peseta = pd.ExcelFile(ruta_bde + 'Tipos de cambio de la peseta frente a las monedas más relevantes (1959-1998) (bde).xlsx')
print('Hojas disponibles:', df_peseta.sheet_names)

Hojas disponibles: ['Índice de hojas', 'Índice de series', 'cuadro_01', 'cuadro_02', 'Notas']


### Índice de todas las series en el dataset

In [4]:
df_peseta_indice = df_peseta.parse('Índice de series', header=3)
df_peseta_indice = df_peseta_indice.dropna(how='all').iloc[:, 1:]
df_peseta_indice.columns = ['clave_serie', 'cod', 'descripcion', 'fecha_inicio', 'fecha_fin', 'frecuencia', 'unidades', 'tipo_serie', 'hoja']
df_peseta_indice = df_peseta_indice.set_index('clave_serie').drop(columns='cod')
df_peseta_indice

,descripcion,fecha_inicio,fecha_fin,frecuencia,unidades,tipo_serie,hoja
clave_serie,,,,,,,
251980,Tipo de cambio. Pesetas por dólar de Estados U...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251981,Tipo de cambio. Pesetas por dólar canadiense. ...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251982,Tipo de cambio. Pesetas por franco francés. Me...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251983,Tipo de cambio. Pesetas por libra esterlina. M...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251984,Tipo de cambio. Pesetas por libra irlandesa . ...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251985,Tipo de cambio. Pesetas por franco suizo. Medi...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251986,Tipo de cambio. Pesetas por 100 francos belgas...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251987,Tipo de cambio. Pesetas por marco alemán. Medi...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01
251988,Tipo de cambio. Pesetas por 100 liras italiana...,1959:01,1998:12,Mensual,Pesetas,Índice,cuadro_01


### Exploración de la estructura del cuadro_01 (mensual)

In [5]:
df_peseta_mensual = df_peseta.parse('cuadro_01', header=2)
df_peseta_mensual= df_peseta_mensual.drop(columns= df_peseta_mensual.columns[np.r_[0, 2:9]], index= 20)
print('Dimensiones cuadro_01 (pesetas por x moneda, meses):', df_peseta_mensual.shape)
df_peseta_mensual

Dimensiones cuadro_01 (pesetas por x moneda, meses): (20, 482)


,Pesetas,Clave serie,1959-01-01 00:00:00,1959-02-01 00:00:00,1959-03-01 00:00:00,1959-04-01 00:00:00,1959-05-01 00:00:00,1959-06-01 00:00:00,1959-07-01 00:00:00,1959-08-01 00:00:00,...,1998-03-01 00:00:00,1998-04-01 00:00:00,1998-05-01 00:00:00,1998-06-01 00:00:00,1998-07-01 00:00:00,1998-08-01 00:00:00,1998-09-01 00:00:00,1998-10-01 00:00:00,1998-11-01 00:00:00,1998-12-01 00:00:00
0,Por dólar de Estados Unidos (2),251980.0,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,48.970000,60.0000,...,154.839500,154.037500,150.679100,152.097100,152.601600,151.784200,144.539300,139.233400,143.153400,142.036000
1,Por dólar canadiense (1),251981.0,43.350000,43.240000,43.020000,43.300000,43.300000,43.300000,50.690000,62.4100,...,109.326000,107.755600,104.291200,103.825200,102.777800,99.074500,94.972400,90.209500,92.970200,92.135700
2,Por franco francés (1),251982.0,8.690000,8.500000,8.500000,8.500000,8.500000,8.500000,9.910000,12.1500,...,25.293500,25.330800,25.333600,25.316300,25.315300,25.317400,25.327800,25.346600,25.360600,25.372400
3,Por libra esterlina (1),251983.0,117.600000,117.600000,117.600000,117.600000,117.600000,117.600000,137.110000,168.0000,...,257.257900,257.520100,246.680300,251.200700,250.849100,247.890300,242.906800,235.906200,237.807500,237.330600
4,Por libra irlandesa (3),251984.0,117.600000,117.600000,117.600000,117.600000,117.600000,117.600000,137.110000,168.0000,...,211.814500,214.000300,213.809100,213.888800,213.486400,213.052000,212.590000,211.956300,211.454600,211.319200
5,Por franco suizo (1),251985.0,9.700000,9.700000,9.700000,9.700000,9.700000,9.700000,11.260000,13.7200,...,104.057100,102.319500,101.985200,101.808100,100.813000,101.560300,103.118800,104.170300,103.282000,104.494300
6,Por 100 francos belgas (1),251986.0,84.000000,84.000000,84.000000,84.000000,84.000000,84.000000,97.940000,120.0000,...,411.025800,411.463200,411.820400,411.506700,411.550400,411.593200,411.690800,411.943700,412.246900,412.507400
7,Por marco alemán (1),251987.0,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,11.660000,14.2800,...,84.791500,84.912800,84.952600,84.882500,84.870600,84.878300,84.927000,84.986900,85.039200,85.085600
8,Por 100 liras italianas (4),251988.0,6.720000,6.720000,6.720000,6.720000,6.720000,6.720000,7.830000,9.6000,...,8.611200,8.596700,8.614300,8.615700,8.607900,8.602400,8.595700,8.590600,8.592700,8.592600
9,Por florín holandés (5),251989.0,11.050000,11.050000,11.050000,11.050000,11.050000,11.050000,12.890000,15.7900,...,75.229900,75.408600,75.385600,75.305800,75.283600,75.263900,75.286600,75.361500,75.422800,75.499300


In [6]:
df_peseta_mensual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Columns: 482 entries, Pesetas to 1998-12-01 00:00:00
dtypes: float64(481), object(1)
memory usage: 75.4+ KB


### Exploración de la estructura del cuadro_02 (diario)

In [7]:
df_peseta_diario = df_peseta.parse('cuadro_02', header=2)
df_peseta_diario= df_peseta_diario.drop(columns= df_peseta_diario.columns[np.r_[0:7]], index= 0)
print('Dimensiones cuadro_2 (dias, pesetas por x moneda):', df_peseta_diario.shape)
df_peseta_diario = df_peseta_diario.rename(columns={'Unnamed: 7': 'fecha'})
df_peseta_diario

Dimensiones cuadro_2 (dias, pesetas por x moneda): (13371, 21)


,fecha,Por dólar de Estados Unidos,Por dólar canadiense,Por franco francés,Por libra esterlina,Por libra irlandesa,Por franco suizo,Por 100 francos belgas,Por marco alemán,Por 100 liras italianas,...,Por corona sueca,Por corona danesa,Por corona noruega,Por marco finlandés,Por 100 chelines austríacos,Por 100 escudos portugueses,Por 100 yenes japoneses(8),Por dólar australiano(9),Por dólar neozelandés (10),Por 100 dracmas griegas (11)
1,1962-05-24 00:00:00,59.869999,…,…,…,…,…,…,…,…,...,…,…,…,…,…,…,…,…,…,…
2,1962-05-25 00:00:00,59.869999,…,…,…,…,…,…,…,…,...,…,…,…,…,…,…,…,…,…,…
3,1962-05-26 00:00:00,…,…,…,…,…,…,…,…,…,...,…,…,…,…,…,…,…,…,…,…
4,1962-05-27 00:00:00,…,…,…,…,…,…,…,…,…,...,…,…,…,…,…,…,…,…,…,…
5,1962-05-28 00:00:00,…,…,…,…,…,…,…,…,…,...,…,…,…,…,…,…,…,…,…,…
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13367,1998-12-27 00:00:00,…,…,…,…,…,…,…,…,…,...,…,…,…,…,…,…,…,…,…,…
13368,1998-12-28 00:00:00,143.076996,92.307999,25.374001,239.481995,211.309998,104.017998,412.503998,85.088997,8.592,...,17.738001,22.363001,18.677999,27.99,1209.400024,82.986,122.792,86.848,74.400002,50.618
13369,1998-12-29 00:00:00,142.464996,92.014,25.370001,239.085007,211.276001,104.293999,412.463989,85.079002,8.593,...,17.632,22.344,18.729,27.986,1209.300049,82.983002,123.721001,87.116997,74.153,50.672001
13370,1998-12-30 00:00:00,142.063004,91.624001,25.368,237.457993,211.289993,103.847,412.463989,85.078003,8.594,...,17.589001,22.341,18.819,27.986,1209.300049,82.981003,123.532997,87.084999,74.696999,50.492001


In [8]:
df_peseta_diario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13371 entries, 1 to 13371
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   fecha                         13371 non-null  object
 1   Por dólar de Estados Unidos   13371 non-null  object
 2   Por dólar canadiense          13371 non-null  object
 3   Por franco francés            13371 non-null  object
 4   Por libra esterlina           13371 non-null  object
 5   Por libra irlandesa           13371 non-null  object
 6   Por franco suizo              13371 non-null  object
 7   Por 100 francos belgas        13371 non-null  object
 8   Por marco alemán              13371 non-null  object
 9   Por 100 liras italianas       13371 non-null  object
 10  Por florín holandés           13371 non-null  object
 11  Por corona sueca              13371 non-null  object
 12  Por corona danesa             13371 non-null  object
 13  Por corona norue

### 1.2 Carga del BCE — Euro foreign exchange reference rates (csv)
Formato ANCHO: una fila por día hábil, columnas = monedas. Cobertura: 1999-2026.

In [9]:
df_euro = pd.read_csv(ruta_bce + 'Euro foreign exchange reference rates (1999-2026) (ecb)eurofxref-hist.csv')
print('Dimensiones del dataset del BCE:', df_euro.shape)
df_euro = df_euro.rename(columns={'Date': 'fecha'}).drop(columns='Unnamed: 42')
df_euro

Dimensiones del dataset del BCE: (7004, 43)


,fecha,USD,JPY,BGN,CYP,CZK,DKK,EEK,GBP,HUF,...,ILS,INR,KRW,MXN,MYR,NZD,PHP,SGD,THB,ZAR
0,2026-05-13,1.1715,184.83,NaN,NaN,24.347,7.4738,NaN,0.86713,358.70,...,3.4066,112.0793,1743.87,20.1774,4.6046,1.9749,71.925,1.4904,37.880,19.2712
1,2026-05-12,1.1738,184.98,NaN,NaN,24.324,7.4729,NaN,0.86803,357.23,...,3.4163,112.2585,1747.64,20.2454,4.6177,1.9743,72.152,1.4937,38.055,19.3990
2,2026-05-11,1.1765,184.87,NaN,NaN,24.333,7.4726,NaN,0.86488,356.03,...,3.4206,112.1385,1732.87,20.2592,4.6148,1.9785,71.683,1.4938,38.136,19.3488
3,2026-05-08,1.1761,184.37,NaN,NaN,24.304,7.4726,NaN,0.86410,356.15,...,3.4167,111.1285,1725.08,20.2716,4.6115,1.9735,71.136,1.4911,37.897,19.3109
4,2026-05-07,1.1770,184.07,NaN,NaN,24.306,7.4726,NaN,0.86410,355.85,...,3.4222,110.9380,1707.40,20.2473,4.6021,1.9686,71.061,1.4898,37.835,19.1354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6999,1999-01-08,1.1659,130.09,NaN,0.58187,34.938,7.4433,15.6466,0.70940,250.15,...,NaN,NaN,1366.73,NaN,NaN,2.1557,NaN,1.9537,NaN,6.7855
7000,1999-01-07,1.1632,129.43,NaN,0.58187,34.886,7.4431,15.6466,0.70585,250.09,...,NaN,NaN,1337.16,NaN,NaN,2.1531,NaN,1.9436,NaN,6.8283
7001,1999-01-06,1.1743,131.42,NaN,0.58200,34.850,7.4452,15.6466,0.70760,250.67,...,NaN,NaN,1359.54,NaN,NaN,2.1890,NaN,1.9699,NaN,6.7307
7002,1999-01-05,1.1790,130.96,NaN,0.58230,34.917,7.4495,15.6466,0.71220,250.80,...,NaN,NaN,1373.01,NaN,NaN,2.2011,NaN,1.9655,NaN,6.7975


In [10]:
df_euro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7004 entries, 0 to 7003
Data columns (total 42 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   fecha   7004 non-null   object 
 1   USD     7004 non-null   float64
 2   JPY     7004 non-null   float64
 3   BGN     6515 non-null   float64
 4   CYP     2304 non-null   float64
 5   CZK     7004 non-null   float64
 6   DKK     7004 non-null   float64
 7   EEK     3074 non-null   float64
 8   GBP     7004 non-null   float64
 9   HUF     7004 non-null   float64
 10  LTL     4097 non-null   float64
 11  LVL     3842 non-null   float64
 12  MTL     2304 non-null   float64
 13  PLN     7004 non-null   float64
 14  ROL     1664 non-null   float64
 15  RON     5340 non-null   float64
 16  SEK     7004 non-null   float64
 17  SIT     2049 non-null   float64
 18  SKK     2560 non-null   float64
 19  CHF     7004 non-null   float64
 20  ISK     4663 non-null   float64
 21  NOK     7004 non-null   float64
 22  

# 2. Limpieza y transformación

### 2.1 Limpieza de tipos de cambio BdE — cuadro_01 (mensual, peseta) y conversión a cotización indirecta

In [11]:
df_peseta_mensual = df_peseta_mensual.drop(columns= 'Clave serie')

meses_peseta = df_peseta_mensual.columns[1:].tolist()
meses_peseta = [pd.Timestamp(fecha) for fecha in meses_peseta]

In [12]:
# Transponer el dataset de pesetas
df_peseta_mensual = df_peseta_mensual.set_index('Pesetas').T
df_peseta_mensual.columns.name = None

# Ponemos los meses como índice y lo renombramos a "fecha"
df_peseta_mensual.index = meses_peseta
df_peseta_mensual.index.name = 'fecha'

df_peseta_mensual = df_peseta_mensual.reset_index()
df_peseta_mensual

,fecha,Por dólar de Estados Unidos (2),Por dólar canadiense (1),Por franco francés (1),Por libra esterlina (1),Por libra irlandesa (3),Por franco suizo (1),Por 100 francos belgas (1),Por marco alemán (1),Por 100 liras italianas (4),...,Por corona sueca (1),Por corona danesa (1),Por corona noruega (1),Por marco finlandés (6),Por 100 chelines austríacos (1),Por 100 escudos portugueses (7),Por 100 yenes japoneses (8),Por dólar australiano (9),Por dólar neozelandés (10),Por 100 dracmas griegas (11)
0,1959-01-01,42.0000,43.3500,8.6900,117.6000,117.6000,9.7000,84.0000,10.0000,6.7200,...,8.1100,6.0800,5.8800,13.1250,160.0000,146.0800,11.6667,47.0398,58.799999,140.0000
1,1959-02-01,42.0000,43.2400,8.5000,117.6000,117.6000,9.7000,84.0000,10.0000,6.7200,...,8.1100,6.0800,5.8800,13.1250,160.0000,146.0800,11.6667,47.0398,58.799999,140.0000
2,1959-03-01,42.0000,43.0200,8.5000,117.6000,117.6000,9.7000,84.0000,10.0000,6.7200,...,8.1100,6.0800,5.8800,13.1250,160.0000,146.0800,11.6667,47.0398,58.799999,140.0000
3,1959-04-01,42.0000,43.3000,8.5000,117.6000,117.6000,9.7000,84.0000,10.0000,6.7200,...,8.1100,6.0800,5.8800,13.1250,160.0000,146.0800,11.6667,47.0398,58.799999,140.0000
4,1959-05-01,42.0000,43.3000,8.5000,117.6000,117.6000,9.7000,84.0000,10.0000,6.7200,...,8.1100,6.0800,5.8800,13.1250,160.0000,146.0800,11.6667,47.0398,58.799999,140.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,1998-08-01,151.7842,99.0745,25.3174,247.8903,213.0520,101.5603,411.5932,84.8783,8.6024,...,18.6664,22.2834,19.6561,27.9112,1206.3191,82.9175,104.9209,89.5333,76.145432,50.4248
476,1998-09-01,144.5393,94.9724,25.3278,242.9068,212.5900,103.1188,411.6908,84.9270,8.5957,...,18.2876,22.2990,19.0806,27.9008,1206.9728,82.8420,107.4244,85.0154,72.889816,49.3606
477,1998-10-01,139.2334,90.2095,25.3466,235.9062,211.9563,104.1703,411.9437,84.9869,8.5906,...,17.7623,22.3499,18.7346,27.9318,1207.9237,82.8620,115.5013,86.0429,72.713058,49.4120
478,1998-11-01,143.1534,92.9702,25.3606,237.8075,211.4546,103.2820,412.2469,85.0392,8.5927,...,17.8649,22.3668,19.1925,27.9648,1208.6998,82.9250,118.7021,91.0244,76.479843,50.6108


In [13]:
df_peseta_mensual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 21 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   fecha                            480 non-null    datetime64[ns]
 1   Por dólar de Estados Unidos (2)  480 non-null    float64       
 2   Por dólar canadiense (1)         480 non-null    float64       
 3   Por franco francés (1)           480 non-null    float64       
 4   Por libra esterlina (1)          480 non-null    float64       
 5   Por libra irlandesa (3)          480 non-null    float64       
 6   Por franco suizo (1)             480 non-null    float64       
 7   Por 100 francos belgas (1)       480 non-null    float64       
 8   Por marco alemán (1)             480 non-null    float64       
 9   Por 100 liras italianas (4)      480 non-null    float64       
 10  Por florín holandés (5)          480 non-null    float64      

In [14]:
# '100 francos belgas' y '100 liras' implican que el valor está por cada 100 unidades
for columna in df_peseta_mensual.columns[1:]:
    if '100' in columna:
        # Dividimos el valor entre 100 para estandarizar
        df_peseta_mensual[columna] = df_peseta_mensual[columna] / 100

        # Eliminamos el "100" del nombre
        nuevo_nombre = columna.replace('100', '')

        # Le cambiamos el nombre a la columna
        df_peseta_mensual = df_peseta_mensual.rename(columns={columna: nuevo_nombre})

df_peseta_mensual.columns

Index(['fecha', 'Por dólar de Estados Unidos (2)', 'Por dólar canadiense (1)',
       'Por franco francés (1)', 'Por libra esterlina (1)',
       'Por libra irlandesa (3)', 'Por franco suizo (1)',
       'Por  francos belgas (1)', 'Por marco alemán (1)',
       'Por  liras italianas (4)', 'Por florín holandés (5)',
       'Por corona sueca (1)', 'Por corona danesa (1)',
       'Por corona noruega (1)', 'Por marco finlandés (6)',
       'Por  chelines austríacos (1)', 'Por  escudos portugueses (7)',
       'Por  yenes japoneses (8)', 'Por dólar australiano (9)',
       'Por dólar neozelandés (10)', 'Por  dracmas griegas (11)'],
      dtype='object')

### Conversión de tipos numéricos, tratamiento de valores nulos y conversión a tipo de cambio directo

In [15]:
# Renombrar columnas a códigos ISO estándar
renombrar_peseta_mensual = {
    'Por dólar de Estados Unidos (2)'  : 'USD_pts',    # Dólar estadounidense
    'Por dólar canadiense (1)'         : 'CAD_pts',    # Dólar canadiense
    'Por franco francés (1)'           : 'FRF_pts',    # Franco francés
    'Por libra esterlina (1)'          : 'GBP_pts',    # Libra esterlina
    'Por libra irlandesa (3)'          : 'IEP_pts',    # Libra irlandesa
    'Por franco suizo (1)'             : 'CHF_pts',    # Franco suizo
    'Por 100 francos belgas (1)'       : 'BEF100_pts', # Franco belga
    'Por marco alemán (1)'             : 'DEM_pts',    # Marco alemán
    'Por 100 liras italianas (4)'      : 'ITL100_pts', # Lira italiana
    'Por florín holandés (5)'          : 'NLG_pts',    # Florín holandés
    'Por corona sueca (1)'             : 'SEK_pts',    # Corona sueca
    'Por corona danesa (1)'            : 'DKK_pts',    # Corona danesa
    'Por corona noruega (1)'           : 'NOK_pts',    # Corona noruega
    'Por marco finlandés (6)'          : 'FIM_pts',    # Marco finlandés
    'Por 100 chelines austríacos (1)'  : 'ATS100_pts', # Chelín austríaco
    'Por 100 escudos portugueses (7)'  : 'PTE100_pts', # Escudo portugués
    'Por 100 yenes japoneses (8)'      : 'JPY100_pts', # Yen japonés
    'Por dólar australiano (9)'        : 'AUD_pts',    # Dólar australiano
    'Por dólar neozelandés (10)'       : 'NZD_pts',    # Dólar neozelandés
    'Por 100 dracmas griegas (11)'     : 'GRD100_pts'  # Dracma griega
}

df_peseta_mensual = df_peseta_mensual.rename(columns=renombrar_peseta_mensual)

In [16]:
# Conversión peseta → cotización indirecta (HEUR_X = 166.386 / X_pts)
columnas_pts = df_peseta_mensual.filter(regex='_pts$').columns
for col in columnas_pts:
    base = col.replace('_pts', '')
    df_peseta_mensual[f'HEUR_{base}'] = 166.386 / df_peseta_mensual[col]
df_peseta_mensual = df_peseta_mensual.drop(columns=list(columnas_pts))
print(f'Convertidas a HEUR_X: {len(columnas_pts)} columnas')

Convertidas a HEUR_X: 14 columnas


In [17]:
df_peseta_mensual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   fecha                         480 non-null    datetime64[ns]
 1   Por  francos belgas (1)       480 non-null    float64       
 2   Por  liras italianas (4)      480 non-null    float64       
 3   Por  chelines austríacos (1)  480 non-null    float64       
 4   Por  escudos portugueses (7)  480 non-null    float64       
 5   Por  yenes japoneses (8)      480 non-null    float64       
 6   Por  dracmas griegas (11)     480 non-null    float64       
 7   HEUR_USD                      480 non-null    float64       
 8   HEUR_CAD                      480 non-null    float64       
 9   HEUR_FRF                      480 non-null    float64       
 10  HEUR_GBP                      480 non-null    float64       
 11  HEUR_IEP                      48

### 2.2 Limpieza de tipos de cambio BdE — cuadro_02 (diario, peseta) y conversión a cotización indirecta

### Conversión de tipos numéricos, tratamiento de valores nulos

In [18]:
for columna in df_peseta_diario.columns[1:]:
    df_peseta_diario[columna] = pd.to_numeric(
        df_peseta_diario[columna].astype(str).str.replace('…', '', regex=False).str.strip(),
        errors='coerce'
    )

# Conversión a formato datetime
df_peseta_diario['fecha'] = pd.to_datetime(df_peseta_diario['fecha'])
df_peseta_diario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13371 entries, 1 to 13371
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   fecha                         13371 non-null  datetime64[ns]
 1   Por dólar de Estados Unidos   8876 non-null   float64       
 2   Por dólar canadiense          7763 non-null   float64       
 3   Por franco francés            7761 non-null   float64       
 4   Por libra esterlina           7763 non-null   float64       
 5   Por libra irlandesa           7763 non-null   float64       
 6   Por franco suizo              7761 non-null   float64       
 7   Por 100 francos belgas        7761 non-null   float64       
 8   Por marco alemán              7761 non-null   float64       
 9   Por 100 liras italianas       7763 non-null   float64       
 10  Por florín holandés           7761 non-null   float64       
 11  Por corona sueca            

In [19]:
# Renombrar columnas a códigos ISO estándar
renombrar_peseta_diario = {
    'Por dólar de Estados Unidos '     : 'USD_pts',    # Dólar estadounidense
    'Por dólar canadiense'             : 'CAD_pts',    # Dólar canadiense
    'Por franco francés'               : 'FRF_pts',    # Franco francés
    'Por libra esterlina'              : 'GBP_pts',    # Libra esterlina
    'Por libra irlandesa'              : 'IEP_pts',    # Libra irlandesa
    'Por franco suizo'                 : 'CHF_pts',    # Franco suizo
    'Por 100 francos belgas'           : 'BEF100_pts', # Franco belga
    'Por marco alemán'                 : 'DEM_pts',    # Marco alemán
    'Por 100 liras italianas'          : 'ITL100_pts', # Lira italiana
    'Por florín holandés'              : 'NLG_pts',    # Florín holandés
    'Por corona sueca'                 : 'SEK_pts',    # Corona sueca
    'Por corona danesa'                : 'DKK_pts',    # Corona danesa
    'Por corona noruega'               : 'NOK_pts',    # Corona noruega
    'Por marco finlandés'              : 'FIM_pts',    # Marco finlandés
    'Por 100 chelines austríacos'      : 'ATS100_pts', # Chelín austríaco
    'Por 100 escudos portugueses'      : 'PTE100_pts', # Escudo portugués
    'Por 100 yenes japoneses(8)'       : 'JPY100_pts', # Yen japonés
    'Por dólar australiano(9)'         : 'AUD_pts',    # Dólar australiano
    'Por dólar neozelandés (10)'       : 'NZD_pts',    # Dólar neozelandés
    'Por 100 dracmas griegas (11)'     : 'GRD100_pts'  # Dracma griega
}

df_peseta_diario = df_peseta_diario.rename(columns=renombrar_peseta_diario)

In [20]:
# '100 francos belgas' y '100 liras' implican que el valor está por cada 100 unidades
columnas_100 = df_peseta_diario.filter(regex='100').columns

# Dividimos el valor entre 100 para estandarizar
df_peseta_diario[columnas_100] = df_peseta_diario[columnas_100] / 100

# Eliminamos el "100" del nombre
df_peseta_diario = df_peseta_diario.rename(columns=lambda x: x.replace('100', ''))

df_peseta_diario.columns

Index(['fecha', 'USD_pts', 'CAD_pts', 'FRF_pts', 'GBP_pts', 'IEP_pts',
       'CHF_pts', 'BEF_pts', 'DEM_pts', 'ITL_pts', 'NLG_pts', 'SEK_pts',
       'DKK_pts', 'NOK_pts', 'FIM_pts', 'ATS_pts', 'PTE_pts', 'JPY_pts',
       'AUD_pts', 'NZD_pts', 'GRD_pts'],
      dtype='object')

In [21]:
# Conversión peseta → cotización indirecta (HEUR_X = 166.386 / X_pts)
columnas_pts = df_peseta_diario.filter(regex='_pts$').columns
for col in columnas_pts:
    base = col.replace('_pts', '')
    df_peseta_diario[f'HEUR_{base}'] = 166.386 / df_peseta_diario[col]
df_peseta_diario = df_peseta_diario.drop(columns=list(columnas_pts))
print(f'Convertidas a HEUR_X: {len(columnas_pts)} columnas')

Convertidas a HEUR_X: 20 columnas


### Añadir columnas temporales: año, trimestre

In [22]:
bde = [df_peseta_mensual, df_peseta_diario]
columnas_temporales = ['fecha', 'año', 'trimestre', 'mes', 'periodo']
for df in bde:
    df['año']       = df['fecha'].dt.year
    df['trimestre'] = df['fecha'].dt.quarter
    df['mes']       = df['fecha'].dt.month
    texto_fecha = df['año'].astype('string') + "Q" + df['trimestre'].astype('string')
    df['periodo'] = texto_fecha

    columnas_valores = [columna for columna in df.columns if columna not in columnas_temporales]
    df = df[columnas_temporales + columnas_valores]

    df = df.sort_values('fecha').reset_index(drop=True)
    print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   fecha                         480 non-null    datetime64[ns]
 1   año                           480 non-null    int32         
 2   trimestre                     480 non-null    int32         
 3   mes                           480 non-null    int32         
 4   periodo                       480 non-null    string        
 5   Por  francos belgas (1)       480 non-null    float64       
 6   Por  liras italianas (4)      480 non-null    float64       
 7   Por  chelines austríacos (1)  480 non-null    float64       
 8   Por  escudos portugueses (7)  480 non-null    float64       
 9   Por  yenes japoneses (8)      480 non-null    float64       
 10  Por  dracmas griegas (11)     480 non-null    float64       
 11  HEUR_USD                      48

### Rellenando gaps dentro del rango (fines de semana, festivos, sin publicación BCE) usando forward-fill

In [23]:
columnas_moneda = [columna for columna in df_peseta_diario.columns if columna not in columnas_temporales]

for columna in columnas_moneda:
    serie = df_peseta_diario[columna]
    primer_dato = serie.first_valid_index()
    ultimo_dato = serie.last_valid_index()

    if primer_dato is None:
        continue

    mask_cobertura = (df_peseta_diario.index >= primer_dato) & (df_peseta_diario.index <= ultimo_dato)
    nulos_dentro = serie[mask_cobertura].isna().sum()

    if nulos_dentro > 0:
        df_peseta_diario.loc[mask_cobertura, columna] = serie[mask_cobertura].ffill(limit=5)

print(f'Nulos restantes (estructurales, fuera de cobertura):')
print(df_peseta_diario[columnas_moneda].isna().sum()[df_peseta_diario[columnas_moneda].isna().sum() > 0])

Nulos restantes (estructurales, fuera de cobertura):
HEUR_USD       19
HEUR_CAD     1704
HEUR_FRF     1704
HEUR_GBP     1704
HEUR_IEP     1704
HEUR_CHF     1704
HEUR_BEF     1704
HEUR_DEM     1704
HEUR_ITL     1704
HEUR_NLG     1704
HEUR_SEK     1704
HEUR_DKK     1704
HEUR_NOK     1704
HEUR_FIM     1704
HEUR_ATS     1704
HEUR_PTE     1704
HEUR_JPY     1704
HEUR_AUD     8317
HEUR_NZD    11211
HEUR_GRD     8866
dtype: int64


In [24]:
df_peseta_diario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13371 entries, 1 to 13371
Data columns (total 25 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   fecha      13371 non-null  datetime64[ns]
 1   HEUR_USD   13352 non-null  float64       
 2   HEUR_CAD   11667 non-null  float64       
 3   HEUR_FRF   11667 non-null  float64       
 4   HEUR_GBP   11667 non-null  float64       
 5   HEUR_IEP   11667 non-null  float64       
 6   HEUR_CHF   11667 non-null  float64       
 7   HEUR_BEF   11667 non-null  float64       
 8   HEUR_DEM   11667 non-null  float64       
 9   HEUR_ITL   11667 non-null  float64       
 10  HEUR_NLG   11667 non-null  float64       
 11  HEUR_SEK   11667 non-null  float64       
 12  HEUR_DKK   11667 non-null  float64       
 13  HEUR_NOK   11667 non-null  float64       
 14  HEUR_FIM   11667 non-null  float64       
 15  HEUR_ATS   11667 non-null  float64       
 16  HEUR_PTE   11667 non-null  float64      

### 2.3 Limpieza de tipos de cambio del euro del BCE

In [25]:
# Datos del BCE en cotización indirecta nativa (divisa por 1 EUR), sin transformación necesaria

In [26]:
df_euro.columns

Index(['fecha', 'USD', 'JPY', 'BGN', 'CYP', 'CZK', 'DKK', 'EEK', 'GBP', 'HUF',
       'LTL', 'LVL', 'MTL', 'PLN', 'ROL', 'RON', 'SEK', 'SIT', 'SKK', 'CHF',
       'ISK', 'NOK', 'HRK', 'RUB', 'TRL', 'TRY', 'AUD', 'BRL', 'CAD', 'CNY',
       'HKD', 'IDR', 'ILS', 'INR', 'KRW', 'MXN', 'MYR', 'NZD', 'PHP', 'SGD',
       'THB', 'ZAR'],
      dtype='object')

In [27]:
# Renombrar columnas a HEUR_X (cotización indirecta: X divisa por 1 HEUR)
renombrar_euro = {
    'USD': 'HEUR_USD',    # Dólar estadounidense
    'JPY': 'HEUR_JPY',    # Yen japonés
    'BGN': 'HEUR_BGN',    # Lev búlgaro
    'CYP': 'HEUR_CYP',    # Libra chipriota
    'CZK': 'HEUR_CZK',    # Corona checa
    'DKK': 'HEUR_DKK',    # Corona danesa
    'EEK': 'HEUR_EEK',    # Corona estonia
    'GBP': 'HEUR_GBP',    # Libra esterlina
    'HUF': 'HEUR_HUF',    # Forinto húngaro
    'LTL': 'HEUR_LTL',    # Litas lituana
    'LVL': 'HEUR_LVL',    # Lats letón
    'MTL': 'HEUR_MTL',    # Lira maltesa
    'PLN': 'HEUR_PLN',    # Zloty polaco
    'ROL': 'HEUR_ROL',    # Leu rumano antiguo
    'RON': 'HEUR_RON',    # Leu rumano nuevo
    'SEK': 'HEUR_SEK',    # Corona sueca
    'SIT': 'HEUR_SIT',    # Tólar esloveno
    'SKK': 'HEUR_SKK',    # Corona eslovaca
    'CHF': 'HEUR_CHF',    # Franco suizo
    'ISK': 'HEUR_ISK',    # Corona islandesa
    'NOK': 'HEUR_NOK',    # Corona noruega
    'HRK': 'HEUR_HRK',    # Kuna croata
    'RUB': 'HEUR_RUB',    # Rublo ruso
    'TRL': 'HEUR_TRL',    # Lira turca antigua
    'TRY': 'HEUR_TRY',    # Lira turca nueva
    'AUD': 'HEUR_AUD',    # Dólar australiano
    'BRL': 'HEUR_BRL',    # Real brasileño
    'CAD': 'HEUR_CAD',    # Dólar canadiense
    'CNY': 'HEUR_CNY',    # Yuan chino
    'HKD': 'HEUR_HKD',    # Dólar de Hong Kong
    'IDR': 'HEUR_IDR',    # Rupia indonesia
    'ILS': 'HEUR_ILS',    # Shékel israelí
    'INR': 'HEUR_INR',    # Rupia india
    'KRW': 'HEUR_KRW',    # Won surcoreano
    'MXN': 'HEUR_MXN',    # Peso mexicano
    'MYR': 'HEUR_MYR',    # Ringgit malayo
    'NZD': 'HEUR_NZD',    # Dólar neozelandés
    'PHP': 'HEUR_PHP',    # Peso filipino
    'SGD': 'HEUR_SGD',    # Dólar de Singapur
    'THB': 'HEUR_THB',    # Baht tailandés
    'ZAR': 'HEUR_ZAR'     # Rand sudafricano
}

df_euro = df_euro.rename(columns=renombrar_euro)

In [28]:
# Conversión a formato datetime
df_euro['fecha'] = pd.to_datetime(df_euro['fecha'])
df_euro.info()
df_euro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7004 entries, 0 to 7003
Data columns (total 42 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   fecha     7004 non-null   datetime64[ns]
 1   HEUR_USD  7004 non-null   float64       
 2   HEUR_JPY  7004 non-null   float64       
 3   HEUR_BGN  6515 non-null   float64       
 4   HEUR_CYP  2304 non-null   float64       
 5   HEUR_CZK  7004 non-null   float64       
 6   HEUR_DKK  7004 non-null   float64       
 7   HEUR_EEK  3074 non-null   float64       
 8   HEUR_GBP  7004 non-null   float64       
 9   HEUR_HUF  7004 non-null   float64       
 10  HEUR_LTL  4097 non-null   float64       
 11  HEUR_LVL  3842 non-null   float64       
 12  HEUR_MTL  2304 non-null   float64       
 13  HEUR_PLN  7004 non-null   float64       
 14  HEUR_ROL  1664 non-null   float64       
 15  HEUR_RON  5340 non-null   float64       
 16  HEUR_SEK  7004 non-null   float64       
 17  HEUR_SIT  2049

In [29]:
# Añadir columnas temporales: año, trimestre, mes
df_euro['año'] = df_euro['fecha'].dt.year
df_euro['trimestre'] = df_euro['fecha'].dt.quarter
df_euro['mes'] = df_euro['fecha'].dt.month
texto_fecha = df_euro['año'].astype('string') + "Q" + df_euro['trimestre'].astype('string')
df_euro['periodo'] = texto_fecha

columnas_valores = [columna for columna in df_euro.columns if columna not in columnas_temporales]
df_euro = df_euro[columnas_temporales + columnas_valores]

df_euro = df_euro.sort_values('fecha').reset_index(drop=True)
df_euro

,fecha,año,trimestre,mes,periodo,HEUR_USD,HEUR_JPY,HEUR_BGN,HEUR_CYP,HEUR_CZK,...,HEUR_ILS,HEUR_INR,HEUR_KRW,HEUR_MXN,HEUR_MYR,HEUR_NZD,HEUR_PHP,HEUR_SGD,HEUR_THB,HEUR_ZAR
0,1999-01-04,1999,1,1,1999Q1,1.1789,133.73,NaN,0.58231,35.107,...,NaN,NaN,1398.59,NaN,NaN,2.2229,NaN,1.9554,NaN,6.9358
1,1999-01-05,1999,1,1,1999Q1,1.1790,130.96,NaN,0.58230,34.917,...,NaN,NaN,1373.01,NaN,NaN,2.2011,NaN,1.9655,NaN,6.7975
2,1999-01-06,1999,1,1,1999Q1,1.1743,131.42,NaN,0.58200,34.850,...,NaN,NaN,1359.54,NaN,NaN,2.1890,NaN,1.9699,NaN,6.7307
3,1999-01-07,1999,1,1,1999Q1,1.1632,129.43,NaN,0.58187,34.886,...,NaN,NaN,1337.16,NaN,NaN,2.1531,NaN,1.9436,NaN,6.8283
4,1999-01-08,1999,1,1,1999Q1,1.1659,130.09,NaN,0.58187,34.938,...,NaN,NaN,1366.73,NaN,NaN,2.1557,NaN,1.9537,NaN,6.7855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6999,2026-05-07,2026,2,5,2026Q2,1.1770,184.07,NaN,NaN,24.306,...,3.4222,110.9380,1707.40,20.2473,4.6021,1.9686,71.061,1.4898,37.835,19.1354
7000,2026-05-08,2026,2,5,2026Q2,1.1761,184.37,NaN,NaN,24.304,...,3.4167,111.1285,1725.08,20.2716,4.6115,1.9735,71.136,1.4911,37.897,19.3109
7001,2026-05-11,2026,2,5,2026Q2,1.1765,184.87,NaN,NaN,24.333,...,3.4206,112.1385,1732.87,20.2592,4.6148,1.9785,71.683,1.4938,38.136,19.3488
7002,2026-05-12,2026,2,5,2026Q2,1.1738,184.98,NaN,NaN,24.324,...,3.4163,112.2585,1747.64,20.2454,4.6177,1.9743,72.152,1.4937,38.055,19.3990


### Rellenando gaps dentro del rango (fines de semana, festivos, sin publicación BCE) usando forward-fill

In [30]:
columnas_moneda = [columna for columna in df_euro.columns if columna not in columnas_temporales]

for columna in columnas_moneda:
    serie = df_euro[columna]
    primer_dato = serie.first_valid_index()
    ultimo_dato = serie.last_valid_index()

    if primer_dato is None:
        continue

    mask_cobertura = (df_euro.index >= primer_dato) & (df_euro.index <= ultimo_dato)
    nulos_dentro = serie[mask_cobertura].isna().sum()

    if nulos_dentro > 0:
        df_euro.loc[mask_cobertura, columna] = serie[mask_cobertura].ffill()

print(f'Nulos restantes (estructurales, fuera de cobertura):')
print(df_euro[columnas_moneda].isna().sum()[df_euro[columnas_moneda].isna().sum() > 0])

Nulos restantes (estructurales, fuera de cobertura):
HEUR_BGN     489
HEUR_CYP    4700
HEUR_EEK    3930
HEUR_LTL    2907
HEUR_LVL    3162
HEUR_MTL    4700
HEUR_ROL    5340
HEUR_RON    1664
HEUR_SIT    4955
HEUR_SKK    4444
HEUR_HRK    2456
HEUR_RUB    2671
HEUR_TRL    5467
HEUR_TRY    1537
HEUR_BRL    2304
HEUR_CNY    1599
HEUR_IDR    1599
HEUR_ILS    3074
HEUR_INR    2560
HEUR_MXN    2304
HEUR_MYR    1599
HEUR_PHP    1599
HEUR_THB    1599
dtype: int64


In [31]:
df_euro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7004 entries, 0 to 7003
Data columns (total 46 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   fecha      7004 non-null   datetime64[ns]
 1   año        7004 non-null   int32         
 2   trimestre  7004 non-null   int32         
 3   mes        7004 non-null   int32         
 4   periodo    7004 non-null   string        
 5   HEUR_USD   7004 non-null   float64       
 6   HEUR_JPY   7004 non-null   float64       
 7   HEUR_BGN   6515 non-null   float64       
 8   HEUR_CYP   2304 non-null   float64       
 9   HEUR_CZK   7004 non-null   float64       
 10  HEUR_DKK   7004 non-null   float64       
 11  HEUR_EEK   3074 non-null   float64       
 12  HEUR_GBP   7004 non-null   float64       
 13  HEUR_HUF   7004 non-null   float64       
 14  HEUR_LTL   4097 non-null   float64       
 15  HEUR_LVL   3842 non-null   float64       
 16  HEUR_MTL   2304 non-null   float64       


# 3. Unión de series

Las dos series tienen **frecuencias distintas** (mensual vs. diaria) y **convenciones de cotización inversas**, que se homogeneizan a cotización indirecta BCE (divisa por 1 EUR) antes de la unión.

- **df_peseta_mensual**: BdE 1959-1998, ya convertida a `HEUR_X`
- **df_peseta_diario**: BdE 1959-1998 diario, ya convertida a `HEUR_X`
- **df_euro**: BCE diario 1999-2026, en `HEUR_X` (formato nativo)

La unión por `concat` empalma ambas en una sola serie continua sin transformación adicional.

In [32]:
df_final = pd.concat([df_peseta_diario, df_euro], ignore_index=True)

df_final = df_final.sort_values('fecha').reset_index(drop=True)
print(df_final.shape)
df_final

(20375, 56)


,fecha,HEUR_USD,HEUR_CAD,HEUR_FRF,HEUR_GBP,HEUR_IEP,HEUR_CHF,HEUR_BEF,HEUR_DEM,HEUR_ITL,...,HEUR_IDR,HEUR_ILS,HEUR_INR,HEUR_KRW,HEUR_MXN,HEUR_MYR,HEUR_PHP,HEUR_SGD,HEUR_THB,HEUR_ZAR
0,1962-05-24,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1962-05-25,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1962-05-26,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1962-05-27,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1962-05-28,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20370,2026-05-07,1.177000,1.6040,NaN,0.86410,NaN,0.9157,NaN,NaN,NaN,...,20360.28,3.4222,110.9380,1707.40,20.2473,4.6021,71.061,1.4898,37.835,19.1354
20371,2026-05-08,1.176100,1.6063,NaN,0.86410,NaN,0.9156,NaN,NaN,NaN,...,20424.09,3.4167,111.1285,1725.08,20.2716,4.6115,71.136,1.4911,37.897,19.3109
20372,2026-05-11,1.176500,1.6079,NaN,0.86488,NaN,0.9163,NaN,NaN,NaN,...,20500.04,3.4206,112.1385,1732.87,20.2592,4.6148,71.683,1.4938,38.136,19.3488
20373,2026-05-12,1.173800,1.6088,NaN,0.86803,NaN,0.9172,NaN,NaN,NaN,...,20538.21,3.4163,112.2585,1747.64,20.2454,4.6177,72.152,1.4937,38.055,19.3990


In [33]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20375 entries, 0 to 20374
Data columns (total 56 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   fecha      20375 non-null  datetime64[ns]
 1   HEUR_USD   20356 non-null  float64       
 2   HEUR_CAD   18671 non-null  float64       
 3   HEUR_FRF   11667 non-null  float64       
 4   HEUR_GBP   18671 non-null  float64       
 5   HEUR_IEP   11667 non-null  float64       
 6   HEUR_CHF   18671 non-null  float64       
 7   HEUR_BEF   11667 non-null  float64       
 8   HEUR_DEM   11667 non-null  float64       
 9   HEUR_ITL   11667 non-null  float64       
 10  HEUR_NLG   11667 non-null  float64       
 11  HEUR_SEK   18671 non-null  float64       
 12  HEUR_DKK   18671 non-null  float64       
 13  HEUR_NOK   18671 non-null  float64       
 14  HEUR_FIM   11667 non-null  float64       
 15  HEUR_ATS   11667 non-null  float64       
 16  HEUR_PTE   11667 non-null  float64      

In [34]:
print(df_final.columns.tolist())

['fecha', 'HEUR_USD', 'HEUR_CAD', 'HEUR_FRF', 'HEUR_GBP', 'HEUR_IEP', 'HEUR_CHF', 'HEUR_BEF', 'HEUR_DEM', 'HEUR_ITL', 'HEUR_NLG', 'HEUR_SEK', 'HEUR_DKK', 'HEUR_NOK', 'HEUR_FIM', 'HEUR_ATS', 'HEUR_PTE', 'HEUR_JPY', 'HEUR_AUD', 'HEUR_NZD', 'HEUR_GRD', 'año', 'trimestre', 'mes', 'periodo', 'HEUR_BGN', 'HEUR_CYP', 'HEUR_CZK', 'HEUR_EEK', 'HEUR_HUF', 'HEUR_LTL', 'HEUR_LVL', 'HEUR_MTL', 'HEUR_PLN', 'HEUR_ROL', 'HEUR_RON', 'HEUR_SIT', 'HEUR_SKK', 'HEUR_ISK', 'HEUR_HRK', 'HEUR_RUB', 'HEUR_TRL', 'HEUR_TRY', 'HEUR_BRL', 'HEUR_CNY', 'HEUR_HKD', 'HEUR_IDR', 'HEUR_ILS', 'HEUR_INR', 'HEUR_KRW', 'HEUR_MXN', 'HEUR_MYR', 'HEUR_PHP', 'HEUR_SGD', 'HEUR_THB', 'HEUR_ZAR']


In [35]:
px.line(df_peseta_diario, x='fecha', y='HEUR_USD',
    title='Tipo de cambio USD: HEUR/USD (BdE peseta convertida)',
    labels={'value': 'Tipo de cambio', 'fecha': 'Fecha'})

In [36]:
px.line(df_euro, x='fecha', y=['HEUR_USD'],
    title='Tipo de cambio USD: HEUR/USD (BCE)',
    labels={'value': 'Tipo de cambio', 'fecha': 'Fecha'}).add_hline(y=1.0, line_dash="dash", line_color="red").show()

#### Cálculo del Euro "histórico" español

Se construye una serie continua 1959–2026 en **cotización indirecta** (divisa por 1 EUR):

- **1959–1998** (BdE peseta): `HEUR_X = 166.386 / X_pts` (aplicado en limpieza, secciones 2.1 / 2.2)
- **1999–2026** (BCE): columna `HEUR_X` directa, ya en formato indirecto

In [37]:
# La unión por concat ya empalma peseta convertida + BCE en HEUR_X
print('df_final unificado en HEUR_X tras el concat')
print(df_final.filter(regex='^HEUR_').describe())

df_final unificado en HEUR_X tras el concat
           HEUR_USD      HEUR_CAD      HEUR_FRF      HEUR_GBP      HEUR_IEP  \
count  20356.000000  18671.000000  11667.000000  18671.000000  11667.000000   
mean       1.700387      1.885798      9.892237      0.890823      1.010294   
std        0.660303      0.523043      2.279941      0.176593      0.154970   
min        0.825200      1.213900      6.305605      0.571100      0.728479   
25%        1.166189      1.478400      8.304145      0.789795      0.919768   
50%        1.372126      1.666560      9.216274      0.862180      0.990806   
75%        2.388982      2.389308     11.954735      0.987703      1.101582   
max        2.997730      3.081822     14.738119      1.556129      1.556129   

           HEUR_CHF      HEUR_BEF      HEUR_DEM      HEUR_ITL      HEUR_NLG  \
count  18671.000000  11667.000000  11667.000000  11667.000000  11667.000000   
mean       3.414891     74.053533      4.627942   1858.413993      4.803801   
std    

In [38]:
# Visualización: USD-Euro "histórico" español
px.line(df_final, x='fecha', y='HEUR_USD',
    title='Tipo de cambio NOMINAL HEUR/USD — Euro "histórico" español',
    labels={'value': 'Tipo de cambio', 'fecha': 'Fecha'}).add_hline(y=1.0, line_dash="dash", line_color="red").show()

# 4. Exportando el dataset final

In [39]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20375 entries, 0 to 20374
Data columns (total 56 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   fecha      20375 non-null  datetime64[ns]
 1   HEUR_USD   20356 non-null  float64       
 2   HEUR_CAD   18671 non-null  float64       
 3   HEUR_FRF   11667 non-null  float64       
 4   HEUR_GBP   18671 non-null  float64       
 5   HEUR_IEP   11667 non-null  float64       
 6   HEUR_CHF   18671 non-null  float64       
 7   HEUR_BEF   11667 non-null  float64       
 8   HEUR_DEM   11667 non-null  float64       
 9   HEUR_ITL   11667 non-null  float64       
 10  HEUR_NLG   11667 non-null  float64       
 11  HEUR_SEK   18671 non-null  float64       
 12  HEUR_DKK   18671 non-null  float64       
 13  HEUR_NOK   18671 non-null  float64       
 14  HEUR_FIM   11667 non-null  float64       
 15  HEUR_ATS   11667 non-null  float64       
 16  HEUR_PTE   11667 non-null  float64      

In [40]:
columnas_moneda = [columna for columna in df_final.columns if columna not in columnas_temporales]

columnas_a_eliminar = []
for columna in columnas_moneda:
    serie = df_final[columna]
    primer_dato = serie.first_valid_index()
    ultimo_dato = serie.last_valid_index()
    primer_año = df_final.loc[primer_dato, 'año']
    ultimo_año = df_final.loc[ultimo_dato, 'año']

    if primer_año > 1975 or ultimo_año < 2025:
        columnas_a_eliminar.append(columna)
        print(f'ELIMINADA {columna}: cobertura {primer_año}–{ultimo_año}')

    else:
        print(f'Correcta {columna}: cobertura {primer_año}–{ultimo_año}')

df_final = df_final.drop(columns=columnas_a_eliminar)
print(f'\nEliminadas {len(columnas_a_eliminar)} columnas. Quedan {len(df_final.columns) - 5} monedas.')


orden = columnas_temporales + [columna for columna in df_final.columns if columna not in columnas_temporales]
df_final = df_final[orden]
df_final

Correcta HEUR_USD: cobertura 1962–2026
Correcta HEUR_CAD: cobertura 1967–2026
ELIMINADA HEUR_FRF: cobertura 1967–1998
Correcta HEUR_GBP: cobertura 1967–2026
ELIMINADA HEUR_IEP: cobertura 1967–1998
Correcta HEUR_CHF: cobertura 1967–2026
ELIMINADA HEUR_BEF: cobertura 1967–1998
ELIMINADA HEUR_DEM: cobertura 1967–1998
ELIMINADA HEUR_ITL: cobertura 1967–1998
ELIMINADA HEUR_NLG: cobertura 1967–1998
Correcta HEUR_SEK: cobertura 1967–2026
Correcta HEUR_DKK: cobertura 1967–2026
Correcta HEUR_NOK: cobertura 1967–2026
ELIMINADA HEUR_FIM: cobertura 1967–1998
ELIMINADA HEUR_ATS: cobertura 1967–1998
ELIMINADA HEUR_PTE: cobertura 1967–1998
Correcta HEUR_JPY: cobertura 1967–2026
ELIMINADA HEUR_AUD: cobertura 1985–2026
ELIMINADA HEUR_NZD: cobertura 1993–2026
ELIMINADA HEUR_GRD: cobertura 1986–1998
ELIMINADA HEUR_BGN: cobertura 2000–2025
ELIMINADA HEUR_CYP: cobertura 1999–2007
ELIMINADA HEUR_CZK: cobertura 1999–2026
ELIMINADA HEUR_EEK: cobertura 1999–2010
ELIMINADA HEUR_HUF: cobertura 1999–2026
ELIMINAD

,fecha,año,trimestre,mes,periodo,HEUR_USD,HEUR_CAD,HEUR_GBP,HEUR_CHF,HEUR_SEK,HEUR_DKK,HEUR_NOK,HEUR_JPY
0,1962-05-24,1962,2,5,1962Q2,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1962-05-25,1962,2,5,1962Q2,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1962-05-26,1962,2,5,1962Q2,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1962-05-27,1962,2,5,1962Q2,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1962-05-28,1962,2,5,1962Q2,2.779121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20370,2026-05-07,2026,2,5,2026Q2,1.177000,1.6040,0.86410,0.9157,10.8250,7.4726,10.8675,184.07
20371,2026-05-08,2026,2,5,2026Q2,1.176100,1.6063,0.86410,0.9156,10.8420,7.4726,10.8215,184.37
20372,2026-05-11,2026,2,5,2026Q2,1.176500,1.6079,0.86488,0.9163,10.8765,7.4726,10.8275,184.87
20373,2026-05-12,2026,2,5,2026Q2,1.173800,1.6088,0.86803,0.9172,10.8935,7.4729,10.7710,184.98


In [41]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20375 entries, 0 to 20374
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   fecha      20375 non-null  datetime64[ns]
 1   año        20375 non-null  int32         
 2   trimestre  20375 non-null  int32         
 3   mes        20375 non-null  int32         
 4   periodo    20375 non-null  string        
 5   HEUR_USD   20356 non-null  float64       
 6   HEUR_CAD   18671 non-null  float64       
 7   HEUR_GBP   18671 non-null  float64       
 8   HEUR_CHF   18671 non-null  float64       
 9   HEUR_SEK   18671 non-null  float64       
 10  HEUR_DKK   18671 non-null  float64       
 11  HEUR_NOK   18671 non-null  float64       
 12  HEUR_JPY   18671 non-null  float64       
dtypes: datetime64[ns](1), float64(8), int32(3), string(1)
memory usage: 1.8 MB


In [42]:
# ── Recortar al rango de tasa de paro y guardar en Datasets ────────────
from pathlib import Path
FECHA_INICIO = '1974-07-01'
FECHA_FIN    = '2025-10-01'
ruta_datasets = Path(r'C:\Users\marco\PycharmProjects\TFM_Marcos\Datasets')

df_final = df_final[(df_final['fecha'] >= FECHA_INICIO) & (df_final['fecha'] <= FECHA_FIN)].copy()
df_final.to_csv(
    ruta_datasets / 'tipo_de_cambio_euro_historico.csv',
    index=False, encoding='utf-8-sig'
)

# 5. Información acerca del dataset final

In [43]:
df_final.attrs['nombre'] = 'df_final'
print('Información acerca del dataset final:')
print(f'{df_final.shape[0]} trimestres × {df_final.shape[1]} columnas | Rango: {df_final["fecha"].min().date()} → {df_final["fecha"].max().date()}')
columnas = df_final.columns.tolist()
print(f'  Columnas: {columnas}')
nulos = df_final.isna().sum()
nulos_col = nulos[nulos > 0]
if len(nulos_col) > 0:
    print(f'ALERTA  {df_final.attrs['nombre']}: nulos en {nulos_col.to_dict()}')
else:
    print(f'CORRECTO  {df_final.attrs['nombre']}: sin valores nulos')
print()

Información acerca del dataset final:
15800 trimestres × 13 columnas | Rango: 1974-07-01 → 2025-10-01
  Columnas: ['fecha', 'año', 'trimestre', 'mes', 'periodo', 'HEUR_USD', 'HEUR_CAD', 'HEUR_GBP', 'HEUR_CHF', 'HEUR_SEK', 'HEUR_DKK', 'HEUR_NOK', 'HEUR_JPY']
CORRECTO  df_final: sin valores nulos



In [44]:
df_final.describe()

,fecha,año,trimestre,mes,HEUR_USD,HEUR_CAD,HEUR_GBP,HEUR_CHF,HEUR_SEK,HEUR_DKK,HEUR_NOK,HEUR_JPY
count,15800,15800.000000,15800.000000,15800.000000,15800.000000,15800.000000,15800.000000,15800.000000,15800.000000,15800.000000,15800.000000,15800.000000
mean,1997-11-09 16:12:21.873417600,1997.356772,2.514937,6.542658,1.443864,1.753503,0.865866,2.284948,9.586594,9.341483,9.612722,240.456770
min,1974-07-01 00:00:00,1974.000000,1.000000,1.000000,0.825200,1.213900,0.571100,0.924200,8.055000,7.147779,7.222500,89.300000
25%,1985-04-23 18:00:00,1985.000000,2.000000,4.000000,1.122075,1.456575,0.766288,1.428500,8.887875,7.451900,8.241875,129.570000
50%,1996-02-15 12:00:00,1996.000000,3.000000,7.000000,1.293100,1.594484,0.845865,1.644122,9.372801,7.563344,9.114297,156.136255
75%,2010-05-03 06:00:00,2010.000000,4.000000,10.000000,1.523008,1.856979,0.901743,2.385103,10.155114,10.444329,10.356083,241.471596
max,2025-10-01 00:00:00,2025.000000,4.000000,12.000000,2.997730,3.081822,1.556129,8.757619,12.959421,18.082487,16.096938,891.672033
std,NaN,14.684517,1.115022,3.437130,0.486830,0.440844,0.177239,1.551759,0.946669,2.573446,1.815055,189.025893


In [45]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15800 entries, 4421 to 20220
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   fecha      15800 non-null  datetime64[ns]
 1   año        15800 non-null  int32         
 2   trimestre  15800 non-null  int32         
 3   mes        15800 non-null  int32         
 4   periodo    15800 non-null  string        
 5   HEUR_USD   15800 non-null  float64       
 6   HEUR_CAD   15800 non-null  float64       
 7   HEUR_GBP   15800 non-null  float64       
 8   HEUR_CHF   15800 non-null  float64       
 9   HEUR_SEK   15800 non-null  float64       
 10  HEUR_DKK   15800 non-null  float64       
 11  HEUR_NOK   15800 non-null  float64       
 12  HEUR_JPY   15800 non-null  float64       
dtypes: datetime64[ns](1), float64(8), int32(3), string(1)
memory usage: 1.5 MB
